In [0]:
"""
To by done by an account admin. 
1. Fill in the CONFIG values below. Use an admin SPN for CLIENT_ID and CLIENT_SECRET
2. Execute on the newest DBR version or serverless
"""
# START: --- CONFIG ----
# TODO: fill this out with your specific information 
## Make sure to set "accounts.cloud.databricks.com", "accounts.azuredatabricks.net" or "accounts.gcp.databricks.com" as appropriate.
HOST = "XXX"

# Current account_id, ie 7a99b43c-b46c-432b-b0a7-814217701909
ACCOUNT_ID = "XXX"

# CLIENT_ID and CLIENT_SECRET come from the account-admin service principal, if you don't have one, follow the Step 1 of this doc:
# https://docs.databricks.com/en/dev-tools/auth/oauth-m2m.html#step-1-create-a-service-principal
# Once you have the service principal ready, generate a CLIENT_SECRET from the Step 3 in the doc mentioned above.
CLIENT_ID = "XXX"

# Ideally, don't store the secret in a raw string form, use SECRETS
# are https://docs.databricks.com/en/security/secrets/index.html
CLIENT_SECRET = "XXX"

# END: --- CONFIG ---

import re
from databricks.sdk import AccountClient
from delta.tables import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, concat_ws

# Add widgets for workspace_id, user_name, target_catalog, and target_schema
dbutils.widgets.text("workspace_id", "")
dbutils.widgets.text("user_name", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")

# Get the values from the widgets
workspace_id = dbutils.widgets.get("workspace_id")
user_name = dbutils.widgets.get("user_name")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")

# Use Databricks SDK and OAuth to authenticate
a = AccountClient(
        host=HOST,
        account_id=ACCOUNT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    )

In [0]:
# Create a function that calls the Databricks API to get the workspace_permissions. 

def create_workspace_permissions_df(workspace_id):

    workspace_permissions = a.workspace_assignment.list(workspace_id=workspace_id)

    # Convert workspace_permissions to a list of dictionaries
    workspace_permissions_list = [perm.as_dict() for perm in workspace_permissions]

    # Create a Spark DataFrame from the list of dictionaries
    workspace_permissions_df = spark.createDataFrame(workspace_permissions_list)

    # Flatten the nested structure and convert permissions to a string
    workspace_permissions_df = workspace_permissions_df.select(
        concat_ws(",", col("permissions")).alias("permissions"),
        col("principal.display_name").alias("principal"),
        col("principal.principal_id").alias("principal_id"),
        col("principal.user_name").alias("user_name")
    )

    # Apply the schema to the DataFrame
    workspace_permissions_df = workspace_permissions_df.select(
        col("permissions").cast("string"),
        col("principal").cast("string"),
        col("principal_id").cast("string"),
        col("user_name").cast("string")
    )

    # Return the results
    return(workspace_permissions_df)

# Create a function that checks if the user has ADMIN permissions this will return a boolean value based on whether or not the user is a workspace admin for the workspace they are requesting. 
def check_admin_permissions(df, user_name):
    # Filter by user_name
    user_permissions = df.filter(col("user_name") == user_name)

    # Check if the user has ADMIN permissions
    if user_permissions.filter(col("permissions").contains("ADMIN")).count() > 0:
        return True
    else:
        return False


In [0]:
def grant_system_table_permissions(principal):
    schemas_df = spark.sql("SHOW SCHEMAS IN system")
    for schema in schemas_df.collect():
    schema_name = schema['databaseName']
    if schema_name not in ('information_schema', 'marketplace'):
        tables_df = spark.sql(f"SHOW TABLES IN system.{schema_name}")
        for table in tables_df.collect():
            table_name = table['tableName']
            # Grant SELECT permission on each table
            spark.sql(f"GRANT SELECT ON TABLE `system`.{schema_name}.{table_name} TO `{principal}`")



In [0]:
# Call the functions to get the permissions for a workspace and check if the user is a workspace admin
workspace_permissions_lookup = create_workspace_permissions_df(workspace_id)
is_admin = check_admin_permissions(workspace_permissions_lookup, user_name)

# Specify the user or group to whom you want to grant permissions; this may be optional depending on config
# TODO: specify your admin SPN
principal = "XXX"  # Replace with admin SPN or admin user
grant_system_table_permissions(principal)

if is_admin:
    # Get the names of all schemas in the system catalog
    schemas_df = spark.sql("SHOW SCHEMAS IN system")
    current_user = 'eda97727-d07d-4f41-bd9b-02ae8a6f1d36'

    # Initialize an empty list to store table names
    all_tables = []

    # Iterate through each schema and get the names of all tables
    # Tables with no workspace_id column are intentionally left out. Additionally, the information_schema and all tables under it are intentionally left out.
    for schema in schemas_df.collect():
        schema_name = schema['databaseName']
        if schema_name not in ('information_schema', 'marketplace'):
            tables_df = spark.sql(f"SHOW TABLES IN system.{schema_name}")
            for table in tables_df.collect():
                table_name = table['tableName']
                if table_name not in ('clean_room_events', 'list_prices', 'node_types'):
                    all_tables.append(f"{schema_name}.{table_name}")
    
    try: 
        # Create the catalog where the views will be stored
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")

        # Create the schema inside the target catalog if it does not exist
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")
        
        target_write_path = f"{target_catalog}.{target_schema}"

        # Grant all privileges on the schema to the user running the notebook (admin) and the user_name (workspace admin). 
        # TODO: Adjust this as needed based on permissions you want to grant. 

        spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {target_catalog} TO `{user_name}`")
        spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG {target_catalog} TO `{principal}`")

        spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {target_write_path} TO `{user_name}`")
        spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {target_write_path} TO `{principal}`")
    except:
        raise Exception("Unable to create the target catalog or schema. Please ensure that the target_catalog and target_schema are entered correctly.")
    for table in all_tables:
        view_name = table.replace('.', '_') + "_view"
        full_view_name = f"{target_write_path}.{view_name}"
        spark.sql(f"""
            CREATE OR REPLACE VIEW {full_view_name} AS
            SELECT * FROM system.{table}
            WHERE workspace_id = '{workspace_id}'
        """)
    spark.sql(f"REVOKE ALL PRIVILEGES ON SCHEMA {target_catalog} FROM `{principal}`")
    spark.sql(f"REVOKE ALL PRIVILEGES ON SCHEMA {target_write_path} FROM `{principal}`")
    

else:
    print("User is not an workspace admin for the workspace they are requesting system tables access for. Please contact your workspace admin to request access and ensure that the username and workspace id are entered correctly.")